# 네이버 언론사 뉴스랭킹 크롤링(일, 기간별) — 기자명 수집 추가판


데일리, 기간별 자료 받기

In [ ]:
#설치패키지

# 🚀 1단계: 필수 패키지 설치 (이것부터 실행하세요!)
!pip install requests beautifulsoup4 pandas openpyxl ipywidgets

# 설치 확인
try:
    import requests, pandas as pd, openpyxl
    from bs4 import BeautifulSoup
    print("✅ 모든 패키지 설치 성공!")
    print("이제 아래 메인 코드를 실행하세요.")
except ImportError as e:
    print(f"❌ 설치 실패: {e}")
    print("다시 실행해주세요!")

In [ ]:
# -*- coding: utf-8 -*-
"""
네이버 언론사별 랭킹 뉴스 수집기 v5.7
- [v5.7] 기사마다 상세페이지에서 기자명·입력시각 수집 (매체 시트에 기자/입력시각 열, 기자별 시트 추가)
- [v5.7] 조회수를 공개하지 않는 매체(조선·동아·중앙·데일리안 등)는 0이 아니라 "미공개"로 표시하고 순위에서 제외
- 일별요약: 날짜 → (매체, 조회수, 순위) 세로형
- 같은 날짜범위 파일이 폴더에 이미 있으면 자동 다운로드 안 함 (폴더 검사 기반)
- Colab 자동 다운로드 중복 방지
- 기본 저장 폴더(downloads) + 사용자 지정 폴더
- Colab에서 윈도우 경로(E:\...) 넣으면 downloads로 대체 + 안내
"""

# ========== 임포트 ==========
import os
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import datetime as dt
import requests
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup
import ipywidgets as widgets
from IPython.display import display, clear_output

# ========== 설정 ==========
MEDIA_CONFIG = {
    "헬스조선": "346",
    "코메디닷컴": "296",
    "뉴스1": "421",
    "연합뉴스": "001",
    "데일리안": "119",
    "서울신문": "081",
    "조선일보": "023",
    "동아일보": "020",
    "중앙일보": "025",
    "YTN": "052",
}

# Colab 파일경로 중복 방지용 (같은 셀에서 여러 번 실행했을 때만 의미 있음)
_DOWNLOAD_HISTORY = set()

# ========== 공통 유틸 ==========
def is_colab():
    try:
        import google.colab  # noqa
        return True
    except:
        return False

def is_windows_style_path(path: str) -> bool:
    return isinstance(path, str) and (":" in path and "\\" in path)

def parse_views(text):
    """조회수 숫자. 페이지에 조회수가 없으면 None (= 미공개, 0회가 아님)"""
    if not text:
        return None
    nums = ''.join(c for c in text if c.isdigit())
    return int(nums) if nums else None

# ========== 기자명 (기사 상세페이지 바이라인) ==========
_REPORTER_SEL = ("em.media_end_head_journalist_name, .media_end_head_journalist_box em.media_end_head_journalist_name, "
                 "span.byline_s, .byline .byline_s, p.byline_p span")

def parse_reporter(soup):
    """상세페이지에서 기자명 추출. 공동 기자는 '·'로 연결 (최대 3명)."""
    names = []
    for tag in soup.select(_REPORTER_SEL):
        t = tag.get_text(" ", strip=True)
        t = re.sub(r"\(.*?\)|\[.*?\]|[\w.+-]+@[\w.-]+", " ", t)
        t = re.sub(r"\s+(기자|특파원|객원기자|선임기자|전문기자|수습기자|논설위원|위원|칼럼니스트|PD|앵커)(?=\s|$).*$", "", t).strip()
        t = re.sub(r"\s+", " ", t)
        if t and 1 < len(t) <= 20 and t not in names:
            names.append(t)
    return "·".join(names[:3])

UA = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
      "Accept-Language": "ko-KR,ko;q=0.9", "Referer": "https://news.naver.com/"}
_META_CACHE = {}   # 링크 → (입력시각, 기자) : 같은 셀을 여러 번 돌려도 다시 조회하지 않음

def fetch_article_meta(link):
    m = re.search(r"/article/(?:\w+/)?(\d{3})/(\d+)", link)
    if not m:
        return "", ""
    key = f"{m.group(1)}_{m.group(2)}"
    if key in _META_CACHE:
        return _META_CACHE[key]
    url = f"https://n.news.naver.com/mnews/article/{m.group(1)}/{m.group(2)}"
    try:
        r = requests.get(url, headers=UA, timeout=10)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, 'html.parser')
        tag = soup.select_one('.media_end_head_info_datestamp_time[data-date-time]')
        pub = (tag.get('data-date-time') or '')[:16] if tag else ''
        res = (pub, parse_reporter(soup))
    except Exception:
        res = ("", "")
    _META_CACHE[key] = res
    return res

# ========== 크롤링 ==========
def fetch_news(date_str, media_name, media_id):
    url = f"https://media.naver.com/press/{media_id}/ranking?date={date_str}"
    try:
        resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, 'html.parser')

        articles = []
        for item in soup.select('a._es_pc_link')[:20]:
            title_elem = item.select_one('strong.list_title')
            view_elem = item.select_one('span.list_view')

            if title_elem:
                title = title_elem.text.strip()
                views = parse_views(view_elem.text if view_elem else "")
                href = item.get('href', '#')
                link = f"https://n.news.naver.com{href}" if href.startswith('/') else href
                articles.append([media_name, title, views, link])

        return articles
    except Exception as e:
        print(f"  ⚠️ {media_name} 수집 실패: {e}")
        return []

def collect_all_data(dates):
    all_data = []
    for date_str in dates:
        print(f"\n📅 {date_str} 수집 중...")
        day_rows = []
        for media_name, media_id in MEDIA_CONFIG.items():
            print(f"  • {media_name}...", end="")
            articles = fetch_news(date_str, media_name, media_id)
            for article in articles:
                day_rows.append([date_str] + article)
            print(f" {len(articles)}건")
            time.sleep(0.5)
        # 기자명·입력시각: 상세페이지 병렬 조회 (6개씩)
        print(f"  ✍️ 기자명 확인 중 ({len(day_rows)}건)...", end="")
        metas = {}
        with ThreadPoolExecutor(max_workers=6) as ex:
            futs = {ex.submit(fetch_article_meta, r[4]): i for i, r in enumerate(day_rows)}
            for f in as_completed(futs):
                metas[futs[f]] = f.result()
        found = 0
        for i, r in enumerate(day_rows):
            pub, rep = metas.get(i, ("", ""))
            found += bool(rep)
            # [날짜, 매체, 제목, 조회수, 기자, 입력시각, 링크]
            all_data.append([r[0], r[1], r[2], r[3], rep, pub, r[4]])
        print(f" {found}건 확인")
    return all_data

# ========== 엑셀 생성 ==========
def create_excel(data, dates, save_dir=None):
    """
    엑셀 생성 후 (filepath, data_key, save_dir) 반환
    data_key = '20251112' 또는 '20251101_20251112'
    """
    original_save_dir = save_dir
    if save_dir is None or save_dir.strip() == "":
        save_dir = "downloads"
    else:
        if is_colab() and is_windows_style_path(save_dir):
            print(f"⚠️ 코랩에서는 윈도우 경로({original_save_dir})로 저장할 수 없습니다.")
            print("👉 /content/drive/... 으로 바꾸거나, 일단 downloads에 저장합니다.")
            save_dir = "downloads"

    try:
        os.makedirs(save_dir, exist_ok=True)
    except OSError:
        print(f"⚠️ 폴더 생성 실패: {save_dir} → downloads로 저장합니다.")
        save_dir = "downloads"
        os.makedirs(save_dir, exist_ok=True)

    data_key = dates[0] if len(dates) == 1 else f"{dates[0]}_{dates[-1]}"
    now = dt.datetime.now()
    filename = f"네이버_랭킹뉴스_{data_key}_{now.strftime('%Y%m%d_%H%M%S')}.xlsx"
    filepath = os.path.join(save_dir, filename)

    df = pd.DataFrame(data, columns=['날짜', '매체', '제목', '조회수', '기자', '입력시각', '링크'])
    df['조회수'] = pd.to_numeric(df['조회수'], errors='coerce')   # None(미공개) → NaN
    grouped = (df.groupby(['날짜', '매체'])['조회수']
                 .agg(lambda x: x.sum() if x.notna().any() else None).reset_index())

    with pd.ExcelWriter(filepath, engine='openpyxl') as writer:
        # 일별요약 (세로형)
        rows = []
        rows.append(['일별요약'])
        all_dates = sorted(grouped['날짜'].unique())

        for d in all_dates:
            rows.append([])
            rows.append(['날짜', d])
            rows.append(['매체', '조회수', '순위'])

            date_df = grouped[grouped['날짜'] == d][['매체', '조회수']].copy()
            got = set(date_df['매체'])
            public = date_df[date_df['조회수'].notna()].sort_values('조회수', ascending=False).reset_index(drop=True)
            for i, r in public.iterrows():
                rows.append([r['매체'], int(r['조회수']), i + 1])
            # 조회수를 공개하지 않는 매체 → 순위 제외
            for _, r in date_df[date_df['조회수'].isna()].iterrows():
                rows.append([r['매체'], '미공개', '-'])
            # 수집 실패(0건) 매체
            for media in MEDIA_CONFIG.keys():
                if media not in got:
                    rows.append([media, '수집실패', '-'])

        summary_df = pd.DataFrame(rows)
        summary_df.to_excel(writer, sheet_name='일별요약', index=False, header=False)

        # 매체별 시트
        for media in MEDIA_CONFIG.keys():
            media_df = df[df['매체'] == media][['날짜', '제목', '조회수', '기자', '입력시각', '링크']]
            if not media_df.empty:
                media_df.to_excel(writer, sheet_name=media[:31], index=False)

        # 기자별 시트 (랭킹 진입 횟수·조회수)
        rep_df = df[df['기자'].astype(str).str.len() > 0]
        if not rep_df.empty:
            rep = (rep_df.groupby(['기자', '매체'])
                        .agg(랭킹진입=('제목', 'count'), 조회수합=('조회수', lambda x: x.sum() if x.notna().any() else None),
                             최고조회수=('조회수', 'max'))
                        .reset_index().sort_values(['조회수합', '랭킹진입'], ascending=False, na_position='last'))
            rep.to_excel(writer, sheet_name='기자별', index=False)

    print(f"\n✅ 파일 생성: {filepath}")
    return filepath, data_key, save_dir

# ========== 다운로드 처리 ==========
def safe_download(filepath, data_key=None, save_dir=None, auto_download=True):
    """
    실제 폴더(save_dir)를 검사해서
    '네이버_랭킹뉴스_{data_key}_*.xlsx'가 이미 하나라도 있으면 자동 다운로드 안 함.
    (지금 만든 파일이 그 '하나'면 내려받고, 이전에 있던 게 있으면 안 내려받음)
    """
    # 로컬 파이썬이면 그냥 끝
    if not is_colab():
        print(f"💾 로컬 저장 완료: {filepath}")
        return True

    if not auto_download:
        print(f"📁 Colab에 저장 완료: {filepath}")
        print("💡 왼쪽 파일탐색기에서 직접 다운로드하세요.")
        return True

    global _DOWNLOAD_HISTORY

    # 파일 경로 기준 1차 방지 (같은 셀 안에서)
    if filepath in _DOWNLOAD_HISTORY:
        print(f"⚠️ 이 파일은 이미 다운로드 요청했습니다: {os.path.basename(filepath)}")
        return True

    # 폴더 기반 중복 검사
    if data_key and save_dir:
        prefix = f"네이버_랭킹뉴스_{data_key}_"
        existing = [
            f for f in os.listdir(save_dir)
            if f.startswith(prefix) and f.endswith(".xlsx")
        ]
        # 이미 이 날짜범위 파일이 있었고, 그게 지금 만든 파일이 아니라면 → 자동다운로드 X
        if len(existing) > 1 or (len(existing) == 1 and os.path.basename(filepath) != existing[0]):
            print(f"⚠️ 이 날짜범위({data_key}) 파일이 이미 폴더에 있어서 자동 다운로드는 건너뜁니다.")
            print("💡 새 파일은 폴더에 저장돼 있어요.")
            return True

    # 여기까지 왔으면 다운로드 진행
    try:
        from google.colab import files
        if not os.path.exists(filepath):
            print(f"❌ 파일을 찾을 수 없습니다: {filepath}")
            return False

        size = os.path.getsize(filepath)
        print(f"📥 다운로드 시작: {filepath} ({size:,} bytes)")
        files.download(filepath)

        _DOWNLOAD_HISTORY.add(filepath)
        print("✅ 다운로드 요청 완료!")
        return True
    except Exception as e:
        print(f"❌ 다운로드 실패: {e}")
        print("💡 수동 다운로드: 좌측 파일 목록에서 직접 받으세요.")
        return False

# ========== 메인 실행 ==========
def run(start_date, end_date=None, save_dir=None, auto_download=True):
    # 날짜 목록 만들기
    if end_date is None:
        dates = [start_date]
    else:
        start_dt = dt.datetime.strptime(start_date, "%Y%m%d")
        end_dt = dt.datetime.strptime(end_date, "%Y%m%d")
        dates = []
        current = start_dt
        while current <= end_dt:
            dates.append(current.strftime("%Y%m%d"))
            current += dt.timedelta(days=1)

    print("="*60)
    print("🚀 네이버 랭킹 뉴스 수집 시작")
    print(f"📅 기간: {dates[0]} ~ {dates[-1]}" if len(dates) > 1 else f"📅 날짜: {dates[0]}")
    print("="*60)

    data = collect_all_data(dates)
    print(f"\n📊 총 {len(data)}건 수집 완료!")

    filepath, data_key, real_save_dir = create_excel(data, dates, save_dir=save_dir)

    safe_download(filepath, data_key=data_key, save_dir=real_save_dir, auto_download=auto_download)

    print("\n" + "="*60)
    print("✨ 모든 작업 완료!")
    print("="*60)

    return filepath

# ========== UI ==========
def create_ui():
    mode = widgets.RadioButtons(
        options=['데일리', '기간'],
        value='데일리',
        description='모드:'
    )

    date_input = widgets.Text(
        value=dt.datetime.today().strftime("%Y%m%d"),
        description='날짜:',
        placeholder='YYYYMMDD'
    )

    start_input = widgets.Text(
        value=(dt.datetime.today() - dt.timedelta(days=6)).strftime("%Y%m%d"),
        description='시작일:',
        placeholder='YYYYMMDD'
    )

    end_input = widgets.Text(
        value=dt.datetime.today().strftime("%Y%m%d"),
        description='종료일:',
        placeholder='YYYYMMDD'
    )

    save_dir_input = widgets.Text(
        value='downloads',
        description='저장폴더:',
        placeholder='예) /content/drive/MyDrive/news'
    )

    auto_dl_toggle = widgets.Checkbox(
        value=True,
        description='Colab 자동다운로드'
    )

    run_btn = widgets.Button(
        description='🚀 실행',
        button_style='success',
        layout=widgets.Layout(width='150px', height='40px')
    )

    output = widgets.Output()

    def on_mode_change(change):
        if mode.value == '데일리':
            date_input.layout.display = 'block'
            start_input.layout.display = 'none'
            end_input.layout.display = 'none'
        else:
            date_input.layout.display = 'none'
            start_input.layout.display = 'block'
            end_input.layout.display = 'block'

    mode.observe(on_mode_change, 'value')
    on_mode_change(None)

    def on_run(b):
        with output:
            clear_output()
            try:
                if mode.value == '데일리':
                    run(
                        date_input.value,
                        save_dir=save_dir_input.value,
                        auto_download=auto_dl_toggle.value
                    )
                else:
                    run(
                        start_input.value,
                        end_input.value,
                        save_dir=save_dir_input.value,
                        auto_download=auto_dl_toggle.value
                    )
            except Exception as e:
                print(f"❌ 오류: {e}")
                import traceback
                traceback.print_exc()

    run_btn.on_click(on_run)

    display(widgets.VBox([
        widgets.HTML("<h2>📊 네이버 랭킹 뉴스 수집기 v5.7</h2><hr>"),
        mode,
        date_input,
        start_input,
        end_input,
        save_dir_input,
        auto_dl_toggle,
        run_btn,
        output
    ]))

# ========== 실행 ==========
if __name__ == "__main__":
    create_ui()

#네이버 월간 콘텐츠 발행_언론사별 크롤링(일간, 월간)

ver.2606 _ 테스트 확인 완료


In [ ]:
!pip install -q requests beautifulsoup4 pandas openpyxl
print("✅ 설치 완료")

In [ ]:
import re, time
from pathlib import Path
from datetime import datetime, timedelta
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlencode
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from IPython.display import display, clear_output
import ipywidgets as w

# ── 매체 코드 ─────────────────────────────────────────────────
MEDIA_CODES = {
    "헬스조선":"346", "코메디닷컴":"296", "뉴스1":"421",
    "연합뉴스":"001", "서울신문":"081", "조선일보":"023",
    "동아일보":"020", "중앙일보":"025", "한겨레":"028",
    "경향신문":"032", "YTN":"052",    "SBS":"055",
    "MBC":"056",     "머니투데이":"008","매일경제":"009",
    "서울경제":"011", "한국경제":"015", "이데일리":"018",
    "뉴시스":"003",
}
MEDIA_LIST = list(MEDIA_CODES.keys())
HEADERS = {
    "User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept":"text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language":"ko-KR,ko;q=0.9",
    "Referer":"https://news.naver.com/",
    "Sec-Fetch-Dest":"document","Sec-Fetch-Mode":"navigate",
    "Upgrade-Insecure-Requests":"1",
}

_yesterday = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")
_DEFAULTS  = ["헬스조선", "코메디닷컴", "조선일보", "YTN", "뉴스1", "서울경제", "연합뉴스"]

# ── UI ────────────────────────────────────────────────────────
매체1_w  = w.Dropdown(options=MEDIA_LIST, value=_DEFAULTS[0], description="매체 1:")
매체2_w  = w.Dropdown(options=MEDIA_LIST, value=_DEFAULTS[1], description="매체 2:")
매체3_w  = w.Dropdown(options=MEDIA_LIST, value=_DEFAULTS[2], description="매체 3:")
매체4_w  = w.Dropdown(options=MEDIA_LIST, value=_DEFAULTS[3], description="매체 4:")
매체5_w  = w.Dropdown(options=MEDIA_LIST, value=_DEFAULTS[4], description="매체 5:")
매체6_w  = w.Dropdown(options=MEDIA_LIST, value=_DEFAULTS[5], description="매체 6:")
매체7_w  = w.Dropdown(options=MEDIA_LIST, value=_DEFAULTS[6], description="매체 7:")
시작일_w = w.Text(value=_yesterday, description="시작일:", placeholder="YYYY-MM-DD")
종료일_w = w.Text(value=_yesterday, description="종료일:", placeholder="YYYY-MM-DD")
실행_btn = w.Button(description="▶  수집 시작", button_style="success",
                    layout=w.Layout(width="200px", height="40px"))
출력     = w.Output()

display(w.VBox([
    w.HTML("<h3>📰 네이버 뉴스 수집기</h3><hr>"),
    w.HTML("<b>📌 매체 선택 (최대 7개 / 중복 선택 시 자동 제거)</b>"),
    매체1_w, 매체2_w, 매체3_w, 매체4_w, 매체5_w, 매체6_w, 매체7_w,
    w.HTML("<hr>"),
    w.HTML("<b>📅 기간 입력 (YYYY-MM-DD) — 하루만 받으려면 시작일=종료일</b>"),
    시작일_w, 종료일_w,
    w.HTML("<hr>"),
    실행_btn, 출력
]))

# ── 유틸 ──────────────────────────────────────────────────────
def clean_cell(text):
    """엑셀 셀에 쓸 수 없는 제어문자 제거"""
    if not text: return ""
    return re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', str(text))

def safe_sheet_name(name):
    """엑셀 시트명 금지 문자 제거 + 31자 제한"""
    name = re.sub(r'[\[\]:*?/\\]', '', str(name))
    name = re.sub(r'[^\w\s\-_()]', '', name, flags=re.UNICODE)
    name = name.strip()
    return name[:31] if name else "Sheet"

def get_selected_media():
    seen, result = set(), []
    for widget in [매체1_w, 매체2_w, 매체3_w, 매체4_w, 매체5_w, 매체6_w, 매체7_w]:
        val = widget.value
        if val not in seen:
            seen.add(val)
            result.append(val)
    return result

def get_date_list(start, end):
    out, cur = [], datetime.strptime(start, "%Y-%m-%d")
    e = datetime.strptime(end, "%Y-%m-%d")
    if cur > e:
        raise ValueError(f"시작일({start})이 종료일({end})보다 늦습니다.")
    while cur <= e:
        out.append(cur.strftime("%Y-%m-%d")); cur += timedelta(days=1)
    return out

def make_session():
    s = requests.Session()
    s.mount("https://", HTTPAdapter(max_retries=Retry(
        total=4, backoff_factor=1.0,
        status_forcelist=[429,500,502,503,504], allowed_methods=["GET"]
    )))
    try: s.get("https://news.naver.com/", headers=HEADERS, timeout=10); time.sleep(0.5)
    except: pass
    return s

def parse_date(raw, fallback=None):
    if not raw: return fallback
    s = re.sub(r"[오전후]", "", str(raw)).strip()
    m = re.search(r"(\d{4})[.\-/](\d{1,2})[.\-/](\d{1,2})", s)
    if m:
        y, mo, d = map(int, m.groups()); return f"{y:04d}-{mo:02d}-{d:02d}"
    m = re.search(r"(\d{1,2})[월.](\d{1,2})[일.]?", s)
    if m:
        mo, d = map(int, m.groups()); return f"{datetime.now().year:04d}-{mo:02d}-{d:02d}"
    return fallback

def collect_day(session, code, date_str):
    clean = date_str.replace("-", "")
    articles, seen = [], set()
    for page in range(1, 101):
        params = {"mode":"LPOD","mid":"sec","oid":code,
                  "listType":"title","date":clean,"page":page}
        url = "https://news.naver.com/main/list.naver?" + urlencode(params)
        try:
            resp = session.get(url, headers=HEADERS, timeout=12)
            resp.raise_for_status()
        except Exception as e:
            print(f"  ⚠️ 오류(page={page}): {e}"); break
        soup = BeautifulSoup(resp.text, "html.parser")
        lists = soup.find_all("ul", class_="type02")
        if not lists: break
        found = 0
        for ul in lists:
            for li in ul.find_all("li"):
                a = li.find("a")
                if not a: continue
                href = a.get("href","").strip()
                if not href or "news.naver.com" not in href: continue
                if f"/{code}/" not in href and f"oid={code}" not in href: continue
                if href in seen: continue
                seen.add(href)
                if href.startswith("/"): href = "https://news.naver.com" + href
                span = li.find("span", class_="date")
                raw  = span.get_text(strip=True) if span else ""
                pub  = parse_date(raw, fallback=date_str)
                if pub == date_str:
                    articles.append({
                        "title":        clean_cell(a.get_text(strip=True)),  # ← 수집 시점에 정제
                        "link":         href,
                        "date_text":    clean_cell(raw),
                        "publish_date": pub
                    })
                    found += 1
        if found == 0: break
        time.sleep(0.25)
    return articles

def save_excel(selected, date_list, all_day_articles):
    now  = datetime.now().strftime("%Y%m%d_%H%M%S")
    s, e = date_list[0].replace("-",""), date_list[-1].replace("-","")
    media_tag = "_".join(selected[:3]) + ("_외" if len(selected) > 3 else "")
    media_tag = re.sub(r'[^\w\-_]', '', media_tag)   # 파일명 안전하게
    fname = f"{media_tag}_{s}_{e}_{now}.xlsx"
    fpath = Path(".") / fname

    with pd.ExcelWriter(fpath, engine="openpyxl") as writer:

        # 전체요약 시트
        ov_rows = []
        for d in date_list:
            wd  = "월화수목금토일"[datetime.strptime(d,"%Y-%m-%d").weekday()]
            row = {"발행일":d, "요일":wd}
            grand = 0
            for m in selected:
                cnt = len(all_day_articles[m].get(d, []))
                row[f"{m}_기사수"] = cnt
                grand += cnt
            row["합계"] = grand
            ov_rows.append(row)
        ov_df = pd.DataFrame(ov_rows)
        stat  = {"발행일":"【통계】","요일":""}
        for m in selected:
            col = f"{m}_기사수"
            stat[col] = f"총 {int(ov_df[col].sum())} / 평균 {ov_df[col].mean():.1f} / 최대 {int(ov_df[col].max())}"
        stat["합계"] = f"총 {int(ov_df['합계'].sum())}"
        pd.concat([ov_df, pd.DataFrame([stat])], ignore_index=True).to_excel(
            writer, sheet_name=safe_sheet_name("전체요약"), index=False)

        # 매체별 요약 + 상세 시트
        for media_name in selected:
            day_arts = all_day_articles[media_name]

            sum_rows, total = [], 0
            for d in date_list:
                arts = day_arts.get(d, [])
                wd   = "월화수목금토일"[datetime.strptime(d,"%Y-%m-%d").weekday()]
                total += len(arts)
                sum_rows.append({"발행일":d, "요일":wd, "기사수":len(arts)})
            sum_df = pd.DataFrame(sum_rows)
            avg  = sum_df["기사수"].mean() if not sum_df.empty else 0
            maxv = sum_df["기사수"].max()  if not sum_df.empty else 0
            stat_row = pd.DataFrame([{"발행일":"【통계】","요일":"",
                                       "기사수":f"총 {total} / 평균 {avg:.1f} / 최대 {maxv}"}])
            pd.concat([sum_df, stat_row], ignore_index=True).to_excel(
                writer, sheet_name=safe_sheet_name(f"{media_name}_요약"), index=False)

            det_rows = []
            for d in date_list:
                for a in day_arts.get(d, []):
                    det_rows.append({
                        "발행일":        a["publish_date"],
                        "제목":          a["title"],        # 이미 collect_day에서 정제됨
                        "링크":          a["link"],
                        "발행일시(raw)": a["date_text"],
                    })
            pd.DataFrame(det_rows).to_excel(
                writer, sheet_name=safe_sheet_name(f"{media_name}_상세"), index=False)

            print(f"  [{media_name}] 총 {total}건")

    return fpath

# ── 버튼 클릭 ─────────────────────────────────────────────────
def on_click(b):
    with 출력:
        clear_output()
        try:
            selected = get_selected_media()
            if not selected:
                print("❌ 매체를 1개 이상 선택해주세요."); return

            date_list = get_date_list(시작일_w.value.strip(), 종료일_w.value.strip())

            print("="*45)
            print(f"  매체 : {', '.join(selected)}")
            print(f"  기간 : {date_list[0]} ~ {date_list[-1]}  ({len(date_list)}일)")
            print("="*45)

            session = make_session()
            all_day_articles = {}

            for media_name in selected:
                code = MEDIA_CODES[media_name]
                print(f"\n📰 [{media_name}] 수집 중...")
                day_articles = {}
                for idx, d in enumerate(date_list, 1):
                    print(f"  [{idx:3d}/{len(date_list)}] {d} ...", end=" ", flush=True)
                    arts = collect_day(session, code, d)
                    print(f"{len(arts)}건")
                    day_articles[d] = arts
                    time.sleep(0.2)
                all_day_articles[media_name] = day_articles

            print("\n📊 엑셀 저장 중...")
            fpath = save_excel(selected, date_list, all_day_articles)

            sheet_info = "전체요약 / " + " / ".join(
                f"{m}_요약 / {m}_상세" for m in selected)
            print(f"\n✅ 저장완료: {fpath.name}")
            print(f"   시트구성: {sheet_info}")

            print("\n⬇️ 다운로드 시작...")
            from google.colab import files as _cf
            _cf.download(str(fpath))
            print("✅ 완료!")

        except ValueError as e:
            print(f"❌ 입력 오류: {e}")
        except Exception as e:
            import traceback
            print(f"❌ 오류: {e}")
            traceback.print_exc()   # ← 앞으로 에러 나면 전체 traceback 출력

실행_btn.on_click(on_click)

In [ ]:
import re, time
from pathlib import Path
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlencode, urlsplit, urlunsplit
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from IPython.display import display, clear_output
import ipywidgets as w

# ── 매체 코드 ─────────────────────────────────────────────────
MEDIA_CODES = {
    "헬스조선":"346", "코메디닷컴":"296", "뉴스1":"421",
    "연합뉴스":"001", "서울신문":"081", "조선일보":"023",
    "동아일보":"020", "중앙일보":"025", "한겨레":"028",
    "경향신문":"032", "YTN":"052",    "SBS":"055",
    "MBC":"056",     "머니투데이":"008","매일경제":"009",
    "서울경제":"011", "한국경제":"015", "이데일리":"018",
    "뉴시스":"003",
}
MEDIA_LIST = list(MEDIA_CODES.keys())
HEADERS = {
    "User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept":"text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language":"ko-KR,ko;q=0.9",
    "Referer":"https://news.naver.com/",
    "Sec-Fetch-Dest":"document","Sec-Fetch-Mode":"navigate",
    "Upgrade-Insecure-Requests":"1",
}

_yesterday = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")
_DEFAULTS  = ["헬스조선", "코메디닷컴", "조선일보", "YTN", "뉴스1", "서울경제", "연합뉴스"]

# ── UI ────────────────────────────────────────────────────────
매체_체크박스 = {
    name: w.Checkbox(value=(name in _DEFAULTS), description=name, indent=False,
                      layout=w.Layout(width="140px"))
    for name in MEDIA_LIST
}

# 4개씩 끊어서 여러 줄로 배치
_체크박스_행 = []
_리스트_items = list(매체_체크박스.values())
for i in range(0, len(_리스트_items), 4):
    _체크박스_행.append(w.HBox(_리스트_items[i:i+4]))

전체선택_btn = w.Button(description="전체 선택", layout=w.Layout(width="100px"))
전체해제_btn = w.Button(description="전체 해제", layout=w.Layout(width="100px"))

def _전체선택(b):
    for cb in 매체_체크박스.values():
        cb.value = True

def _전체해제(b):
    for cb in 매체_체크박스.values():
        cb.value = False

전체선택_btn.on_click(_전체선택)
전체해제_btn.on_click(_전체해제)

시작일_w = w.Text(value=_yesterday, description="시작일:", placeholder="YYYY-MM-DD")
종료일_w = w.Text(value=_yesterday, description="종료일:", placeholder="YYYY-MM-DD")
디버그_w = w.Checkbox(value=False, description="🔍 디버그 로그 출력 (원인 진단용, 첫 페이지만)")
실행_btn = w.Button(description="▶  수집 시작", button_style="success",
                    layout=w.Layout(width="200px", height="40px"))
출력     = w.Output()

display(w.VBox([
    w.HTML("<h3>📰 네이버 뉴스 수집기 (발행일시순 정렬)</h3><hr>"),
    w.HTML("<b>📌 매체 선택 (원하는 만큼 자유롭게 체크)</b>"),
    w.HBox([전체선택_btn, 전체해제_btn]),
    *_체크박스_행,
    w.HTML("<hr>"),
    w.HTML("<b>📅 기간 입력 (YYYY-MM-DD) — 하루만 받으려면 시작일=종료일</b>"),
    시작일_w, 종료일_w,
    디버그_w,
    w.HTML("<hr>"),
    실행_btn, 출력
]))

# ── 유틸 ──────────────────────────────────────────────────────
def is_colab():
    try:
        import google.colab  # noqa
        return True
    except Exception:
        return False


def clean_cell(text):
    """엑셀 셀에 쓸 수 없는 제어문자 제거"""
    if not text: return ""
    return re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', str(text))

def safe_sheet_name(name):
    """엑셀 시트명 금지 문자 제거 + 31자 제한"""
    name = re.sub(r'[\[\]:*?/\\]', '', str(name))
    name = re.sub(r'[^\w\s\-_()]', '', name, flags=re.UNICODE)
    name = name.strip()
    return name[:31] if name else "Sheet"

def get_selected_media():
    # MEDIA_LIST 순서를 유지해서 체크된 매체만 반환 (개수 제한 없음)
    return [name for name in MEDIA_LIST if 매체_체크박스[name].value]

def get_date_list(start, end):
    out, cur = [], datetime.strptime(start, "%Y-%m-%d")
    e = datetime.strptime(end, "%Y-%m-%d")
    if cur > e:
        raise ValueError(f"시작일({start})이 종료일({end})보다 늦습니다.")
    while cur <= e:
        out.append(cur.strftime("%Y-%m-%d")); cur += timedelta(days=1)
    return out

def make_session():
    s = requests.Session()
    s.mount("https://", HTTPAdapter(
        max_retries=Retry(
            total=4, backoff_factor=1.0,
            status_forcelist=[429,500,502,503,504], allowed_methods=["GET"]
        ),
        pool_connections=20, pool_maxsize=20
    ))
    try: s.get("https://news.naver.com/", headers=HEADERS, timeout=10); time.sleep(0.5)
    except: pass
    return s


def parse_pub_datetime(raw):
    """
    페이지에 표시된 날짜/시간 원문(raw)을 실제 발행일/발행시각으로 변환.
    네이버 목록 페이지는 절대 날짜("2026.07.26.")가 아니라 상대 시간
    ("3일전", "어제", "2시간전", "5분전", "방금전") 을 쓰는 경우가 많다.
    상대 시간은 반드시 '실제 오늘 날짜'(datetime.now()) 기준으로 역산해야 하며,
    수집 대상으로 요청한 날짜(date_str)를 기준으로 계산하면 안 된다 — 실제 방송/작성
    시점과 무관하게 요청 날짜로 라벨링되어버리는 오류가 생기기 때문.
    파싱 실패 시 (None, None)을 반환하고, 호출부는 이 기사를 버려야 한다
    (엉뚱한 날짜에 잘못 끼워넣는 것보다 누락이 낫다).
    반환: (발행일 'YYYY-MM-DD', 발행시각 'HH:MM' or None)
    """
    if not raw:
        return None, None

    s = str(raw).strip()
    today = datetime.now().date()

    def _extract_time(text):
        tm = re.search(r'(\d{1,2}):(\d{2})', text)
        if not tm:
            return None
        hh, mm = int(tm.group(1)), int(tm.group(2))
        if '오후' in text and hh < 12: hh += 12
        if '오전' in text and hh == 12: hh = 0
        return f"{hh:02d}:{mm:02d}"

    # 1) 절대 날짜: 2026-07-26 / 2026.07.26. 등
    dm = re.search(r'(\d{4})[.\-/](\d{1,2})[.\-/](\d{1,2})', s)
    if dm:
        y, mo, d = map(int, dm.groups())
        return f"{y:04d}-{mo:02d}-{d:02d}", _extract_time(s)

    # 2) 방금전 / N분전 / N시간전 → 오늘
    if re.search(r'방금', s) or re.search(r'\d+\s*분\s*전', s) or re.search(r'\d+\s*시간\s*전', s):
        return today.strftime("%Y-%m-%d"), None

    # 3) 어제
    if '어제' in s:
        return (today - timedelta(days=1)).strftime("%Y-%m-%d"), None

    # 4) N일전  ← 오늘 붙여주신 데이터가 이 케이스였음
    dm3 = re.search(r'(\d+)\s*일\s*전', s)
    if dm3:
        n = int(dm3.group(1))
        return (today - timedelta(days=n)).strftime("%Y-%m-%d"), None

    # 5) 월.일 (연도 없는 절대 날짜) → 올해로 가정, 미래 날짜가 되면 작년으로 보정
    dm2 = re.search(r'(\d{1,2})[월.](\d{1,2})[일.]?', s)
    if dm2:
        mo, d = map(int, dm2.groups())
        try:
            candidate = datetime(today.year, mo, d).date()
            if candidate > today:
                candidate = datetime(today.year - 1, mo, d).date()
            return candidate.strftime("%Y-%m-%d"), None
        except ValueError:
            return None, None

    # 6) 시:분만 있는 경우 → 오늘자로 간주
    time_only = _extract_time(s)
    if time_only:
        return today.strftime("%Y-%m-%d"), time_only

    return None, None


def _time_sort_key(time_str):
    """정렬용: 발행시각이 없으면 맨 뒤로 보낸다."""
    if not time_str:
        return (1, 0)
    try:
        hh, mm = map(int, time_str.split(':'))
        return (0, hh * 60 + mm)
    except Exception:
        return (1, 0)


def _canonical_href(href):
    """href에서 추적용 쿼리스트링(?rc=N&ntype=RANKING 등)을 제거해 깔끔한 링크로 만든다."""
    try:
        parts = urlsplit(href)
        return urlunsplit((parts.scheme, parts.netloc, parts.path, '', ''))
    except Exception:
        return href


def _article_key(href):
    """
    기사 고유 식별자(oid/aid)를 뽑아서 중복 판정 키로 쓴다.
    같은 기사라도 페이지 위치에 따라 쿼리스트링(?rc=N&ntype=RANKING 등)이
    다르게 붙어서 내려올 수 있는데, href 문자열 그대로 비교하면 이걸
    서로 다른 기사로 착각해서 여러 날짜에 중복으로 잡히는 문제가 생긴다.
    """
    m = re.search(r'/article/(\d+)/(\d+)', href)
    if m:
        return f"{m.group(1)}_{m.group(2)}"
    m = re.search(r'oid=(\d+).*?aid=(\d+)', href)
    if m:
        return f"{m.group(1)}_{m.group(2)}"
    # 위 패턴에 안 걸리면 쿼리스트링만 제거한 URL 자체를 키로 사용
    return _canonical_href(href)


# ========== 기자명 (기사 상세페이지 바이라인) ==========
_REPORTER_SEL = ("em.media_end_head_journalist_name, .media_end_head_journalist_box em.media_end_head_journalist_name, "
                 "span.byline_s, .byline .byline_s, p.byline_p span")

def parse_reporter(soup):
    """상세페이지에서 기자명 추출. 공동 기자는 '·'로 연결 (최대 3명)."""
    names = []
    for tag in soup.select(_REPORTER_SEL):
        t = tag.get_text(" ", strip=True)
        t = re.sub(r"\(.*?\)|\[.*?\]|[\w.+-]+@[\w.-]+", " ", t)
        t = re.sub(r"\s+(기자|특파원|객원기자|선임기자|전문기자|수습기자|논설위원|위원|칼럼니스트|PD|앵커)(?=\s|$).*$", "", t).strip()
        t = re.sub(r"\s+", " ", t)
        if t and 1 < len(t) <= 20 and t not in names:
            names.append(t)
    return "·".join(names[:3])


def fetch_article_pubdate(session, link, timeout=10):
    """
    기사 상세페이지에서 정확한 발행일시를 가져온다.
    목록 페이지의 'N일전' 라벨은 하루 단위로만 뭉뚱그려져 있어서 날짜 경계에서
    부정확하므로, 최종 확정은 반드시 이 함수(상세페이지)를 거친다.
    [추가] 같은 페이지에서 기자명도 함께 읽는다.
    반환: (발행일 'YYYY-MM-DD', 발행시각 'HH:MM', 기자) 또는 (None, None, "")
    """
    try:
        resp = session.get(link, headers=HEADERS, timeout=timeout)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")

        raw_time = None
        time_tag = soup.select_one('.media_end_head_info_datestamp_time')
        if time_tag and time_tag.get('data-date-time'):
            raw_time = time_tag.get('data-date-time')
        if not raw_time:
            meta_time = soup.select_one('meta[property="article:published_time"]')
            if meta_time and meta_time.get('content'):
                raw_time = meta_time.get('content')

        reporter = parse_reporter(soup)
        if not raw_time:
            return None, None, reporter

        raw_time = raw_time.strip()
        if 'T' in raw_time:
            date_part, time_part = raw_time.split('T', 1)
            time_part = time_part[:5]
        else:
            parts = raw_time.split(' ')
            date_part = parts[0]
            time_part = parts[1][:5] if len(parts) > 1 else None

        date_part = date_part.replace('-', '')
        if len(date_part) != 8 or not date_part.isdigit():
            return None, None, reporter

        return f"{date_part[:4]}-{date_part[4:6]}-{date_part[6:8]}", time_part, reporter

    except Exception:
        return None, None, ""


def collect_range(session, code, date_list, max_pages=None, debug=False):
    """
    매체(oid=code) 하나에 대해 2단계로 수집한다.

    1단계 (후보 수집, 목록 페이지): 최신순 목록을 페이지별로 순회하며 'N일전'
    같은 대략적인 라벨로 "이 기간 근처"에 있는 기사들을 후보로만 추려낸다.
    라벨이 하루 단위로 뭉뚱그려져 있어 경계에서 틀릴 수 있으므로, 요청 기간
    앞뒤로 하루씩 여유(margin)를 둔다.

    2단계 (정확한 날짜 확정, 상세페이지): 후보로 뽑힌 기사마다 상세페이지에
    직접 들어가서 정확한 발행일시를 가져오고, 그걸로 최종 날짜를 확정한다.
    이 단계에서 실제로는 요청 기간 밖인 것으로 밝혀지면 버린다.

    주의: 목록 페이지의 date= 파라미터가 날짜별로 결과를 제대로 걸러주지
    않는 것처럼 보여서, 서버의 date 필터를 신뢰하지 않고 최신순 목록을
    그대로 받아 우리가 직접 날짜를 계산한다.

    max_pages: 지정 안 하면 요청 기간(일수)에 비례해서 자동으로 늘어난다.
    발행량이 많은 매체를 여러 날짜에 걸쳐 조회하면 가장 오래된 날짜까지
    도달하는 데 페이지가 많이 필요할 수 있기 때문 (예: 하루 100건씩 내는
    매체를 7일치 조회하면 700건 이상 페이지를 넘겨야 가장 오래된 날짜에 닿음).
    """
    earliest, latest = min(date_list), max(date_list)
    earliest_dt = datetime.strptime(earliest, "%Y-%m-%d").date()
    latest_dt = datetime.strptime(latest, "%Y-%m-%d").date()
    margin = timedelta(days=1)
    window_start = (earliest_dt - margin).strftime("%Y-%m-%d")
    window_end = (latest_dt + margin).strftime("%Y-%m-%d")

    if max_pages is None:
        max_pages = max(300, len(date_list) * 150)

    anchor_str = latest.replace("-", "")  # 요청 기간의 최신 날짜를 기준으로 페이지 요청

    buckets = {d: [] for d in date_list}
    seen = set()
    prev_page_hrefs = None
    unparsed = 0
    stale_page_streak = 0
    reached_target = False  # 요청 기간보다 오래된 지점까지 실제로 도달했는지
    candidates = []  # [(clean_link, title, raw), ...] — 정확한 날짜는 2단계에서 확정

    # ── 1단계: 목록 페이지에서 후보 수집 ─────────────────────
    for page in range(1, max_pages + 1):
        params = {"mode":"LPOD","mid":"sec","oid":code,
                  "listType":"title","date":anchor_str,"page":page}
        url = "https://news.naver.com/main/list.naver?" + urlencode(params)
        try:
            resp = session.get(url, headers=HEADERS, timeout=12)
            resp.raise_for_status()
        except Exception as e:
            print(f"  ⚠️ 오류(page={page}): {e}"); break
        soup = BeautifulSoup(resp.text, "html.parser")
        lists = soup.find_all("ul", class_="type02")

        if debug and page <= 3:
            print(f"\n    [DEBUG p{page}] status={resp.status_code} len={len(resp.text)} ul.type02개수={len(lists)}")
            if lists:
                sample_li = lists[0].find_all("li")[:2]
                for i, li in enumerate(sample_li):
                    a = li.find("a")
                    span = li.find("span", class_="date")
                    print(f"      샘플{i}: href={a.get('href') if a else None}")
                    print(f"              date_span존재={span is not None} raw_text={repr(span.get_text(strip=True)) if span else None}")

        if not lists:
            if debug: print(f"    [DEBUG p{page}] ul.type02를 못 찾음 → 중단")
            reached_target = True
            break

        page_hrefs = set()
        new_on_page = 0
        page_has_any_dated = False
        page_all_older_than_window = True

        for ul in lists:
            for li in ul.find_all("li"):
                a = li.find("a")
                if not a: continue
                href = a.get("href","").strip()
                if not href or "news.naver.com" not in href: continue
                if f"/{code}/" not in href and f"oid={code}" not in href: continue
                key = _article_key(href)
                page_hrefs.add(key)
                if key in seen: continue
                seen.add(key)
                new_on_page += 1
                if href.startswith("/"): href = "https://news.naver.com" + href
                clean_link = _canonical_href(href)
                span = li.find("span", class_="date")
                raw  = span.get_text(strip=True) if span else ""
                rough_date, _ = parse_pub_datetime(raw)

                if rough_date is None:
                    unparsed += 1
                    continue

                page_has_any_dated = True
                if rough_date >= window_start:
                    page_all_older_than_window = False
                if window_start <= rough_date <= window_end:
                    candidates.append((clean_link, clean_cell(a.get_text(strip=True)), clean_cell(raw)))

        if prev_page_hrefs is not None and page_hrefs and page_hrefs == prev_page_hrefs:
            if debug: print(f"    [DEBUG p{page}] 이전 페이지와 href가 100% 동일 → 페이지 반복으로 판단, 중단")
            reached_target = True
            break
        prev_page_hrefs = page_hrefs

        if new_on_page == 0:
            if debug: print(f"    [DEBUG p{page}] 새로 추가된 기사 0건 → 중단")
            reached_target = True
            break

        if page_has_any_dated and page_all_older_than_window:
            stale_page_streak += 1
        else:
            stale_page_streak = 0

        if stale_page_streak >= 2:
            if debug: print(f"    [DEBUG p{page}] 요청 기간(여유 포함 {window_start}~)보다 오래된 기사만 연속 2페이지 → 중단")
            reached_target = True
            break

        if debug and page <= 5:
            print(f"    [DEBUG p{page}] 신규 {new_on_page}건, 후보 누적 {len(candidates)}건, stale_streak={stale_page_streak}")

        time.sleep(0.25)

    if not reached_target:
        print(f"    ⚠️ 페이지 상한({max_pages}페이지)까지 다 봤는데도 요청 기간({earliest}~)의 끝에 도달하지 못했습니다.")
        print(f"       → 발행량이 많은 매체라 이 기간 내 더 오래된 날짜(특히 {earliest})는 데이터가 누락됐을 수 있습니다.")
        print(f"       → 확실히 하려면 날짜 범위를 좁혀서 나눠 돌리는 걸 추천합니다.")

    if unparsed and debug:
        print(f"    [DEBUG] 1단계에서 대략적인 날짜 파싱 실패로 제외된 기사: {unparsed}건")

    # ── 2단계: 후보들의 정확한 발행일시를 상세페이지에서 확정 (병렬 처리) ──
    total_candidates = len(candidates)
    print(f"    📄 상세페이지 확인 대상: {total_candidates}건 (병렬로 처리합니다, 시간이 걸릴 수 있어요)")
    if debug:
        print(f"    [DEBUG] 후보 {total_candidates}건 → 상세페이지에서 정확한 발행일시 확인 중...")

    exact_fail = 0
    done = 0
    MAX_WORKERS = 6  # 너무 높이면 차단될 수 있어 6 정도로 제한

    if total_candidates:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            future_to_item = {
                executor.submit(fetch_article_pubdate, session, link): (link, title, raw)
                for link, title, raw in candidates
            }
            for future in as_completed(future_to_item):
                link, title, raw = future_to_item[future]
                done += 1
                try:
                    pub_date, pub_time, reporter = future.result()
                except Exception:
                    pub_date, pub_time, reporter = None, None, ""

                if pub_date is None:
                    exact_fail += 1
                else:
                    if earliest <= pub_date <= latest and pub_date in buckets:
                        buckets[pub_date].append({
                            "title": title,
                            "link": link,
                            "date_text": raw,
                            "publish_date": pub_date,
                            "publish_time": pub_time,
                            "reporter": reporter,
                        })

                if done % 20 == 0 or done == total_candidates:
                    print(f"      진행 {done}/{total_candidates}건 처리...", flush=True)

    if exact_fail and debug:
        print(f"    [DEBUG] 상세페이지 조회 실패로 제외된 기사: {exact_fail}건")

    for d in buckets:
        buckets[d].sort(key=lambda a: _time_sort_key(a["publish_time"]))

    return buckets

def save_excel(selected, date_list, all_day_articles):
    now  = datetime.now().strftime("%Y%m%d_%H%M%S")
    s, e = date_list[0].replace("-",""), date_list[-1].replace("-","")
    media_tag = "_".join(selected[:3]) + ("_외" if len(selected) > 3 else "")
    media_tag = re.sub(r'[^\w\-_]', '', media_tag)   # 파일명 안전하게
    fname = f"{media_tag}_{s}_{e}_{now}.xlsx"
    fpath = Path(".") / fname

    with pd.ExcelWriter(fpath, engine="openpyxl") as writer:

        # 전체요약 시트 (발행일 기준 매체별 일일 건수)
        ov_rows = []
        for d in date_list:
            wd  = "월화수목금토일"[datetime.strptime(d,"%Y-%m-%d").weekday()]
            row = {"발행일":d, "요일":wd}
            grand = 0
            for m in selected:
                cnt = len(all_day_articles[m].get(d, []))
                row[f"{m}_기사수"] = cnt
                grand += cnt
            row["합계"] = grand
            ov_rows.append(row)
        ov_df = pd.DataFrame(ov_rows)
        stat  = {"발행일":"【통계】","요일":""}
        for m in selected:
            col = f"{m}_기사수"
            stat[col] = f"총 {int(ov_df[col].sum())} / 평균 {ov_df[col].mean():.1f} / 최대 {int(ov_df[col].max())}"
        stat["합계"] = f"총 {int(ov_df['합계'].sum())}"
        pd.concat([ov_df, pd.DataFrame([stat])], ignore_index=True).to_excel(
            writer, sheet_name=safe_sheet_name("전체요약"), index=False)

        # 매체별 요약(일별 발행 건수) + 상세(발행일시순) 시트
        for media_name in selected:
            day_arts = all_day_articles[media_name]

            sum_rows, total = [], 0
            for d in date_list:
                arts = day_arts.get(d, [])
                wd   = "월화수목금토일"[datetime.strptime(d,"%Y-%m-%d").weekday()]
                total += len(arts)
                sum_rows.append({"발행일":d, "요일":wd, "발행건수":len(arts)})
            sum_df = pd.DataFrame(sum_rows)
            avg  = sum_df["발행건수"].mean() if not sum_df.empty else 0
            maxv = sum_df["발행건수"].max()  if not sum_df.empty else 0
            stat_row = pd.DataFrame([{"발행일":"【통계】","요일":"",
                                       "발행건수":f"총 {total} / 평균 {avg:.1f} / 최대 {maxv}"}])
            pd.concat([sum_df, stat_row], ignore_index=True).to_excel(
                writer, sheet_name=safe_sheet_name(f"{media_name}_요약"), index=False)

            # 상세: 날짜(오름차순) → 그 안에서 발행시각(오름차순)
            det_rows = []
            for d in date_list:
                day_list = sorted(day_arts.get(d, []), key=lambda a: _time_sort_key(a["publish_time"]))
                for a in day_list:
                    det_rows.append({
                        "발행일":        a["publish_date"],
                        "발행시각":      a["publish_time"] or "",
                        "제목":          a["title"],
                        "기자":          a.get("reporter", ""),
                        "링크":          a["link"],
                        "발행일시(raw)": a["date_text"],
                    })
            pd.DataFrame(det_rows).to_excel(
                writer, sheet_name=safe_sheet_name(f"{media_name}_상세"), index=False)

            print(f"  [{media_name}] 총 {total}건")

        # [추가] 기자별 시트: 매체·기자별 발행 건수
        rep_rows = {}
        for media_name in selected:
            for d in date_list:
                for a in all_day_articles[media_name].get(d, []):
                    rep = a.get("reporter") or ""
                    if rep:
                        k = (media_name, rep)
                        rep_rows[k] = rep_rows.get(k, 0) + 1
        if rep_rows:
            rep_df = pd.DataFrame([{"매체": m, "기자": r, "발행건수": n} for (m, r), n in rep_rows.items()])
            rep_df.sort_values(["매체", "발행건수"], ascending=[True, False]).to_excel(
                writer, sheet_name="기자별", index=False)
            print(f"  [기자별] {len(rep_rows)}명")

    return fpath

# ── 버튼 클릭 ─────────────────────────────────────────────────
def on_click(b):
    with 출력:
        clear_output()
        try:
            selected = get_selected_media()
            if not selected:
                print("❌ 매체를 1개 이상 선택해주세요."); return

            date_list = get_date_list(시작일_w.value.strip(), 종료일_w.value.strip())

            print("="*45)
            print(f"  매체 : {', '.join(selected)}")
            print(f"  기간 : {date_list[0]} ~ {date_list[-1]}  ({len(date_list)}일)")
            print("="*45)

            debug = 디버그_w.value
            session = make_session()
            all_day_articles = {}

            for media_name in selected:
                code = MEDIA_CODES[media_name]
                print(f"\n📰 [{media_name}] 수집 중...")
                buckets = collect_range(session, code, date_list, debug=debug)
                for d in date_list:
                    print(f"  {d}: {len(buckets[d])}건")
                all_day_articles[media_name] = buckets
                time.sleep(0.2)

            print("\n📊 엑셀 저장 중...")
            fpath = save_excel(selected, date_list, all_day_articles)

            sheet_info = "전체요약 / " + " / ".join(
                f"{m}_요약 / {m}_상세" for m in selected)
            print(f"\n✅ 저장완료: {fpath.name}")
            print(f"   시트구성: {sheet_info}")

            if is_colab():
                print("\n⬇️ 다운로드 시작...")
                from google.colab import files as _cf
                _cf.download(str(fpath))
                print("✅ 완료!")
            else:
                print(f"\n💾 로컬 저장 완료: {fpath.resolve()}")

        except ValueError as e:
            print(f"❌ 입력 오류: {e}")
        except Exception as e:
            import traceback
            print(f"❌ 오류: {e}")
            traceback.print_exc()   # ← 앞으로 에러 나면 전체 traceback 출력

실행_btn.on_click(on_click)

# 기존 엑셀에 기자명 채우기
이미 받아둔 엑셀(기자 열 없음)을 올리면 기자명을 채운 파일로 돌려줍니다. 새로 수집하는 파일은 위 셀들에서 기자명이 자동으로 들어갑니다.

In [ ]:
# ========== 기존 엑셀에 기자명 채우기 ==========
# 이미 받아둔 랭킹/발행 엑셀(기자 열이 없는 파일)을 올리면, 링크마다 상세페이지를 확인해서
# '기자'(랭킹 파일은 '입력시각'도) 열을 채운 새 파일(파일명_기자.xlsx)을 만들어 내려받습니다.
# 1만 건 기준 15~20분 정도 걸립니다.
import io, re, zipfile, requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from bs4 import BeautifulSoup
from openpyxl import load_workbook
from google.colab import files

_UA = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
       "Accept-Language": "ko-KR,ko;q=0.9", "Referer": "https://news.naver.com/"}
_S = requests.Session()
_S.mount("https://", requests.adapters.HTTPAdapter(pool_maxsize=12, max_retries=3))

# ========== 기자명 (기사 상세페이지 바이라인) ==========
_REPORTER_SEL = ("em.media_end_head_journalist_name, .media_end_head_journalist_box em.media_end_head_journalist_name, "
                 "span.byline_s, .byline .byline_s, p.byline_p span")

def parse_reporter(soup):
    """상세페이지에서 기자명 추출. 공동 기자는 '·'로 연결 (최대 3명)."""
    names = []
    for tag in soup.select(_REPORTER_SEL):
        t = tag.get_text(" ", strip=True)
        t = re.sub(r"\(.*?\)|\[.*?\]|[\w.+-]+@[\w.-]+", " ", t)
        t = re.sub(r"\s+(기자|특파원|객원기자|선임기자|전문기자|수습기자|논설위원|위원|칼럼니스트|PD|앵커)(?=\s|$).*$", "", t).strip()
        t = re.sub(r"\s+", " ", t)
        if t and 1 < len(t) <= 20 and t not in names:
            names.append(t)
    return "·".join(names[:3])

def _key(link):
    m = re.search(r"/article/(?:\w+/)?(\d{3})/(\d{6,})", str(link or ""))
    return (m.group(1), m.group(2)) if m else None

def _meta(oid, aid):
    try:
        r = _S.get(f"https://n.news.naver.com/mnews/article/{oid}/{aid}", headers=_UA, timeout=12)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, "html.parser")
        t = soup.select_one(".media_end_head_info_datestamp_time[data-date-time]")
        return ((t.get("data-date-time") or "")[:16] if t else ""), parse_reporter(soup)
    except Exception:
        return None

print("📂 기자명을 채울 엑셀 파일을 모두 선택하세요 (여러 개 가능)")
up = files.upload()
books = {n: load_workbook(io.BytesIO(b)) for n, b in up.items()}

keys = set()
for wb in books.values():
    for ws in wb.worksheets:
        head = [c.value for c in ws[1]]
        if "링크" in head:
            li = head.index("링크")
            for row in ws.iter_rows(min_row=2, values_only=True):
                k = _key(row[li]) if li < len(row) else None
                if k: keys.add(k)
keys = list(keys)
print(f"🔎 고유 기사 {len(keys)}건 확인 시작")
info, fail = {}, 0
with ThreadPoolExecutor(8) as ex:
    futs = {ex.submit(_meta, *k): k for k in keys}
    for i, f in enumerate(as_completed(futs), 1):
        res = f.result()
        if res is None: fail += 1
        else: info[futs[f]] = res
        if i % 500 == 0: print(f"   {i}/{len(keys)}")
print(f"✅ 확인 {len(info)}건 (기자명 {sum(1 for v in info.values() if v[1])}건) / 실패 {fail}건")

outs = []
for name, wb in books.items():
    for ws in wb.worksheets:
        head = [c.value for c in ws[1]]
        if "링크" not in head:
            continue
        li = head.index("링크") + 1
        is_rank = "조회수" in head and "발행시각" not in head
        add = [h for h in (["기자", "입력시각"] if is_rank else ["기자"]) if h not in head]
        for h in add:                       # 링크 열 앞에 새 열 삽입
            ws.insert_cols(li); ws.cell(1, li, h); li += 1
        head = [c.value for c in ws[1]]
        ci = {h: head.index(h) + 1 for h in ("기자", "입력시각") if h in head}
        for r in range(2, ws.max_row + 1):
            k = _key(ws.cell(r, li).value)
            if k and k in info:
                pub, rep = info[k]
                if "기자" in ci and not ws.cell(r, ci["기자"]).value: ws.cell(r, ci["기자"], rep)
                if "입력시각" in ci and not ws.cell(r, ci["입력시각"]).value: ws.cell(r, ci["입력시각"], pub)
    out = re.sub(r"\.xlsx$", "", name) + "_기자.xlsx"
    wb.save(out); outs.append(out)

if len(outs) == 1:
    files.download(outs[0])
else:
    with zipfile.ZipFile("기자명_채운_엑셀.zip", "w") as z:
        for o in outs: z.write(o)
    files.download("기자명_채운_엑셀.zip")
print(f"📥 {len(outs)}개 파일 저장 완료")

# 자동 메일 보고서 설정
전날 데이터를 엑셀로 정리한 뒤 `eh_son@chosun.com`으로 보내는 기능입니다.
처음 한 번 실행하면 현재 로그인한 Google 계정의 Gmail 발송 권한을 승인해야 합니다.

In [ ]:
# ========== 랭킹 + 전체 발행 파일 이메일 발송 ==========
!pip install -q google-api-python-client google-auth-httplib2 google-auth-oauthlib

import base64
import mimetypes
from email.message import EmailMessage
from pathlib import Path
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

REPORT_TO = 'eh_son@chosun.com'
AUTOMATION_MEDIA = ['헬스조선', '코메디닷컴', '조선일보', 'YTN', '뉴스1', '서울경제', '연합뉴스']
KST = ZoneInfo('Asia/Seoul')


def _attach_file(message, filepath):
    filepath = Path(filepath)
    mime_type, _ = mimetypes.guess_type(str(filepath))
    maintype, subtype = (mime_type or 'application/octet-stream').split('/', 1)
    with filepath.open('rb') as file:
        message.add_attachment(
            file.read(),
            maintype=maintype,
            subtype=subtype,
            filename=filepath.name,
        )


def send_two_report_email(ranking_filepath, publication_filepath, report_date,
                          selected, all_day_articles, test_mode=True):
    from google.colab import auth
    import google.auth
    from googleapiclient.discovery import build

    print('🔐 Gmail 발송 권한 확인 중...')
    auth.authenticate_user()
    credentials, _ = google.auth.default(
        scopes=['https://www.googleapis.com/auth/gmail.send']
    )
    service = build('gmail', 'v1', credentials=credentials, cache_discovery=False)

    summary = []
    for media_name in selected:
        count = len(all_day_articles[media_name].get(report_date, []))
        summary.append(f'• {media_name}: {count}건')

    subject_prefix = '[테스트] ' if test_mode else '[자동보고] '
    message = EmailMessage()
    message['To'] = REPORT_TO
    message['Subject'] = f'{subject_prefix}네이버 건강뉴스 수집 결과 - {report_date}'
    message.set_content(
        f'전날({report_date}) 네이버 건강뉴스 수집 결과입니다.\n\n'
        + '\n'.join(summary)
        + '\n\n첨부파일: 네이버 랭킹 뉴스, 매체별 전체 발행 데이터'
    )

    _attach_file(message, ranking_filepath)
    _attach_file(message, publication_filepath)

    encoded = base64.urlsafe_b64encode(message.as_bytes()).decode()
    service.users().messages().send(
        userId='me',
        body={'raw': encoded}
    ).execute()
    print(f'✅ {REPORT_TO} 메일 발송 완료')
    print(f'   첨부 1: {Path(ranking_filepath).name}')
    print(f'   첨부 2: {Path(publication_filepath).name}')


def run_daily_email_report(test_mode=True):
    required = ['run', 'MEDIA_CONFIG', 'MEDIA_CODES', 'make_session', 'collect_range', 'save_excel']
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError(
            '먼저 위쪽 셀을 순서대로 실행해 주세요. 빠진 항목: ' + ', '.join(missing)
        )

    report_date = (datetime.now(KST) - timedelta(days=1)).strftime('%Y-%m-%d')
    ranking_date = report_date.replace('-', '')
    selected = [name for name in AUTOMATION_MEDIA if name in MEDIA_CODES]
    if not selected:
        raise RuntimeError('자동화 대상 매체가 없습니다.')

    print(f'📅 전날 데이터 수집: {report_date}')
    print('📊 [1/2] 네이버 랭킹 뉴스 수집 중...')
    ranking_filepath = run(
        ranking_date,
        save_dir='downloads',
        auto_download=False,
    )

    print('📰 [2/2] 전체 발행 데이터 수집 중...')
    session = make_session()
    date_list = [report_date]
    all_day_articles = {}
    for media_name in selected:
        code = MEDIA_CODES[media_name]
        print(f'   [{media_name}] 수집 중...')
        all_day_articles[media_name] = collect_range(session, code, date_list)

    publication_filepath = save_excel(selected, date_list, all_day_articles)
    send_two_report_email(
        ranking_filepath,
        publication_filepath,
        report_date,
        selected,
        all_day_articles,
        test_mode=test_mode,
    )
    return ranking_filepath, publication_filepath


# 테스트할 때는 True로 실행합니다. 예약 실행도 같은 함수가 두 파일을 만들어 발송합니다.
RUN_EMAIL_NOW = True
if RUN_EMAIL_NOW:
    run_daily_email_report(test_mode=True)
else:
    print('ℹ️ 이메일 실행 대기 중입니다.')
